# Newcastle United — 2024/25 vs 2025/26

A **real Premier League results** notebook focused on responsible, explainable analytics. Use it alongside the HTML dashboard and Excel workbook. Original match data: [OpenFootball](https://github.com/openfootball/football.json) (CC0-1.0).

## 1. Reuse the cleaned, checked-in 76-match table

You can run this notebook from the `notebooks/` directory or the repository root. No SQL required.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / "data" / "processed").exists():
    root = root.parent
matches = pd.read_csv(root / "data/processed/newcastle_matches.csv", parse_dates=["date"])
summary = pd.read_csv(root / "data/processed/season_summary.csv")
summary

## 2. Validate the totals

Our checks assert 38 games for each full season, match results and no invented records.

In [ ]:
assert matches.groupby("season").size().to_dict() == {"2024-25":38, "2025-26":38}
assert summary.set_index("season")["points"].to_dict() == {"2024-25":66, "2025-26":49}
assert summary.set_index("season")["wins"].to_dict() == {"2024-25":20, "2025-26":14}
print("All key totals verified")

## 3. Compare league goals and wins

In [ ]:
summary.set_index("season")[["wins", "goals_for", "goals_against", "points"]].T.plot(kind="bar", figsize=(10,5), rot=0, title="Newcastle performance comparison")
plt.ylabel("Count / points")
plt.tight_layout()
plt.show()

## 4. Identify *where* the decline appeared

Comparing the same 19 home fixtures and 19 away fixtures per season avoids mixing the denominators.

In [ ]:
breakdown = matches.groupby(["season", "venue"]).agg(played=("points", "size"), points=("points", "sum"), goals_for=("goals_for", "sum"), goals_against=("goals_against", "sum"), clean_sheets=("clean_sheet", "sum")).reset_index()
breakdown["ppg"] = breakdown.points / breakdown.played
breakdown

## 5. Look at chronological form

The 2024/25 Matchweek 29 game was played after Matchweek 32, so use `match_number` for cumulative comparisons, not naive matchweek ordering.

In [ ]:
for season, grp in matches.groupby("season"):
    grp=grp.sort_values("match_number")
    plt.plot(grp.match_number, grp.cumulative_points, label=season, linewidth=2)
plt.title("Cumulative league points over 38 games")
plt.xlabel("Actual fixture sequence")
plt.ylabel("Cumulative points")
plt.legend()
plt.show()

## 6. Interpret without overclaiming

The data shows away goals and points dropping strongly while home defending weakened. **It does not contain xG, shots, possession or injury information**. Without additional valid sources, it cannot prove why the team declined. Review [CASE_STUDY.md](../docs/CASE_STUDY.md) for the written interpretation.